# Synthetic Data Experiments

In [1]:
# Optional: in case the imports for the local folders/files fail
import os

if os.path.basename(os.getcwd()) == "analysis":
    os.chdir("../")

In [2]:
import numpy as np
import pandas as pd
from analysis import models

## Synthetic Data Generation

In [3]:
# We generate our synthetic data


def generate_synthetic_data(
    n_steps=10000,
    omega_x=1.5,
    alpha_x=0.6,
    lambda_x=0.8,
    omega_y=1.2,
    alpha_y=0.5,
    lambda_y=0.65,
    gamma=0.5,
    seed=42,
):
    """Generates two nonlinear coupled series for model validation"""
    rng = np.random.default_rng(seed)
    X = np.zeros(n_steps)
    Y = np.zeros(n_steps)

    x_prev = 0.0
    y_prev = 0.0

    for t in range(n_steps):
        # X's variance = its own volatility clustering + its own sign-sensitive quirk
        var_x = omega_x + alpha_x * x_prev**2 + lambda_x * np.tanh(x_prev)
        x_t = np.sqrt(var_x) * rng.normal()

        # Y's variance = its own clustering, sign-sensitive quirk (independent of X's),
        # + an X squared term
        var_y = (
            omega_y
            + alpha_y * y_prev**2
            + lambda_y * np.tanh(y_prev)
            + gamma * x_prev**2
        )
        y_t = np.sqrt(var_y) * rng.normal()

        X[t] = x_t
        Y[t] = y_t

        x_prev = x_t
        y_prev = y_t

    return X, Y

In [4]:
X, Y = generate_synthetic_data()

In [5]:
X  # sanity check

array([ 0.37320068,  1.02595153, -3.23505495, ..., -1.79221674,
       -1.32711971,  0.72461819], shape=(10000,))

In [6]:
Y  # sanity check

array([-1.13924551,  1.10866036, -2.20341109, ...,  3.09252878,
        1.8403774 ,  2.46246345], shape=(10000,))

## Results

In [7]:
def summarize_synthetic_validation(x, y, max_lag=10, n_bins=8):
    """
    Runs all 5 models in both directions on a single synthetic
    (x, y) realization, summarizing each into the lag where its statistic
    peaks and what its value is"""
    pairs_fwd = [(x, y)]
    pairs_rev = [(y, x)]
    rows = []

    for direction, (a, b) in [("forward (X->Y)", (x, y)), ("reverse (Y->X)", (y, x))]:
        table = models.compute_lagged_correlations(a, b, max_lag=max_lag)
        table = table[table["lag"] >= 1]
        idx = table["correlation"].abs().idxmax()
        rows.append(
            {
                "method": "Lagged Correlation",
                "direction": direction,
                "best_lag": int(table.loc[idx, "lag"]),
                "peak_stat": table.loc[idx, "correlation"],
            }
        )

    for direction, (a, b) in [("forward (X->Y)", (x, y)), ("reverse (Y->X)", (y, x))]:
        table = models.compute_granger_causality(a, b, max_lag=max_lag)
        idx = table["f_stat"].idxmax()
        rows.append(
            {
                "method": "Granger Causality",
                "direction": direction,
                "best_lag": int(table.loc[idx, "lag"]),
                "peak_stat": table.loc[idx, "f_stat"],
                "p_value_at_peak": table.loc[idx, "p_value"],
            }
        )

    for direction, pairs in [
        ("forward (X->Y)", pairs_fwd),
        ("reverse (Y->X)", pairs_rev),
    ]:
        table = models.compute_tdmi_pooled(pairs, max_lag=max_lag, n_bins=n_bins)
        idx = table["tdmi"].idxmax()
        rows.append(
            {
                "method": "TDMI",
                "direction": direction,
                "best_lag": int(table.loc[idx, "lag"]),
                "peak_stat": table.loc[idx, "tdmi"],
            }
        )

    for direction, pairs in [
        ("forward (X->Y)", pairs_fwd),
        ("reverse (Y->X)", pairs_rev),
    ]:
        table = models.compute_te_pooled(pairs, max_lag=max_lag, n_bins=n_bins)
        idx = table["transfer_entropy"].idxmax()
        rows.append(
            {
                "method": "TE",
                "direction": direction,
                "best_lag": int(table.loc[idx, "lag"]),
                "peak_stat": table.loc[idx, "transfer_entropy"],
            }
        )

    for direction, pairs in [
        ("forward (X->Y)", pairs_fwd),
        ("reverse (Y->X)", pairs_rev),
    ]:
        table = models.compute_te_nl_pooled(pairs, max_lag=max_lag, n_bins=n_bins)
        idx = table["te_nl"].idxmax()
        rows.append(
            {
                "method": "TE_NL",
                "direction": direction,
                "best_lag": int(table.loc[idx, "lag"]),
                "peak_stat": table.loc[idx, "te_nl"],
            }
        )

    return pd.DataFrame(rows)

In [8]:
summarize_synthetic_validation(X, Y)

,method,direction,best_lag,peak_stat,p_value_at_peak
0,Lagged Correlation,forward (X->Y),1,-0.030625,NaN
1,Lagged Correlation,reverse (Y->X),2,-0.029834,NaN
2,Granger Causality,forward (X->Y),1,9.411747,0.002162
3,Granger Causality,reverse (Y->X),2,5.105012,0.006082
4,TDMI,forward (X->Y),1,0.027803,NaN
5,TDMI,reverse (Y->X),7,0.003298,NaN
6,TE,forward (X->Y),1,0.048626,NaN
7,TE,reverse (Y->X),6,0.022088,NaN
8,TE_NL,forward (X->Y),1,0.048121,NaN
9,TE_NL,reverse (Y->X),6,0.021135,NaN
